对称量化+正确率测试( export_test_vector.ipynb 的翻版)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

# ==========================================
# 1. 基础网络定义
# ==========================================
class HardwareFriendlyCNN(nn.Module):
    def __init__(self):
        super(HardwareFriendlyCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * 8 * 8, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# ==========================================
# 2. 对称伪量化函数 (用于测试 10000 张图片的准确率)
# ==========================================
def sym_fake_quantize(tensor):
    """对称伪量化：强制 ZP=0，模拟硬件截断误差后还原为 Float"""
    with torch.no_grad():
        max_abs = tensor.abs().max().item()
        scale = max_abs / 127.0 if max_abs > 0 else 1.0
        
        # 模拟硬件的 INT8
        q_tensor = torch.round(tensor / scale)
        q_tensor = torch.clamp(q_tensor, -128, 127)
        
        # 还原回浮点数，但带上了对称截断的误差
        dq_tensor = q_tensor * scale
        return dq_tensor

# 包装成带量化感知的模型
class SymPTQ_Model(nn.Module):
    def __init__(self, original_model):
        super(SymPTQ_Model, self).__init__()
        self.orig = original_model

    def forward(self, x):
        x = sym_fake_quantize(x)

        # Conv1
        w1_fq = sym_fake_quantize(self.orig.conv1.weight)
        b1_fq = sym_fake_quantize(self.orig.conv1.bias)
        x = F.conv2d(x, w1_fq, b1_fq, stride=self.orig.conv1.stride, padding=self.orig.conv1.padding)
        x = sym_fake_quantize(x)
        x = self.orig.pool(self.orig.relu(x))

        # Conv2
        w2_fq = sym_fake_quantize(self.orig.conv2.weight)
        b2_fq = sym_fake_quantize(self.orig.conv2.bias)
        x = F.conv2d(x, w2_fq, b2_fq, stride=self.orig.conv2.stride, padding=self.orig.conv2.padding)
        x = sym_fake_quantize(x)
        x = self.orig.pool(self.orig.relu(x))

        # FC
        x = torch.flatten(x, 1)
        x = sym_fake_quantize(x)
        w_fc_fq = sym_fake_quantize(self.orig.fc.weight)
        b_fc_fq = sym_fake_quantize(self.orig.fc.bias)
        x = F.linear(x, w_fc_fq, b_fc_fq)
        
        return x

# ==========================================
# 3. 硬件真实的对称量化函数与导出工具
# ==========================================
def sym_quantize(tensor):
    max_abs = tensor.abs().max().item()
    scale = max_abs / 127.0 if max_abs > 0 else 1.0
    q_tensor = torch.round(tensor / scale)
    q_tensor = torch.clamp(q_tensor, -128, 127).to(torch.int8)
    return q_tensor, scale

def export_c_array(tensor, name, file_handle, c_type="int8_t"):
    flat_data = tensor.flatten().tolist()
    file_handle.write(f"const {c_type} {name}[{len(flat_data)}] = {{\n    ")
    for i, val in enumerate(flat_data):
        file_handle.write(f"{val}, ")
        if (i + 1) % 16 == 0:
            file_handle.write("\n    ")
    file_handle.write("\n};\n\n")

# ==========================================
# 4. 主流程
# ==========================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("===== 阶段 1: 加载模型与数据集 =====")
    
    model = HardwareFriendlyCNN()
    model.load_state_dict(torch.load('my_hardware_cnn_fp32_69p.pth', weights_only=True))
    model.eval()

    transform = transforms.Compose([transforms.ToTensor()])
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

    print("\n===== 阶段 2: 评估对称量化 (ZP=0) 整体准确率 =====")
    sym_model = SymPTQ_Model(model).to(device)
    sym_model.eval()

    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = sym_model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"📉 对称量化 (ZP=0) 评估完毕，准确率: {accuracy:.2f} %")
    
    print("\n===== 阶段 3: 提取 C 语言测试向量 (单张图片) =====")
    # 取第 0 张图
    img_fp32, label = testset[0]
    img_fp32 = img_fp32.unsqueeze(0) 

    w_fp32 = model.conv1.weight.data.cpu()
    b_fp32 = model.conv1.bias.data.cpu()

    img_q, s_in = sym_quantize(img_fp32)
    w_q, s_w = sym_quantize(w_fp32)
    
    s_bias = s_in * s_w
    b_q = torch.round(b_fp32 / s_bias).to(torch.int32)

    out_fp32 = F.conv2d(img_fp32, w_fp32, b_fp32, padding=1)
    _, s_out = sym_quantize(out_fp32)
    out_zp = 0

    M = (s_in * s_w) / s_out
    shift = 16 
    multiplier = int(round(M * (1 << shift)))

    # 模拟精确的 C 语言算术
    exact_mac_int32 = F.conv2d(img_q.float(), w_q.float(), bias=b_q.float(), padding=1).to(torch.int32)
    scaled_acc = (exact_mac_int32 * multiplier) >> shift
    expected_out_q = torch.clamp(scaled_acc + out_zp, -128, 127).to(torch.int8)

    with open("test_vector_2.h", "w") as f:
        f.write("#ifndef TEST_VECTOR_H\n#define TEST_VECTOR_H\n\n")
        f.write("#include <stdint.h>\n\n")
        f.write(f"// 宏定义提取参数，请直接供 main.c 使用\n")
        f.write(f"#define REAL_MULTIPLIER {multiplier}\n")
        f.write(f"#define REAL_SHIFT {shift}\n")
        f.write(f"#define REAL_OUT_ZP {out_zp}\n\n")
        
        export_c_array(img_q, "test_input_img", f, "int8_t")
        export_c_array(w_q, "test_conv1_weight", f, "int8_t")
        export_c_array(b_q, "test_conv1_bias", f, "int32_t")
        export_c_array(expected_out_q, "expected_conv1_output", f, "int8_t")
        
        f.write("#endif // TEST_VECTOR_H\n")

    print(f"✅ 完成！test_vector.h 已生成，Multiplier={multiplier}")

if __name__ == "__main__":
    main()

===== 阶段 1: 加载模型与数据集 =====

===== 阶段 2: 评估对称量化 (ZP=0) 整体准确率 =====
📉 对称量化 (ZP=0) 评估完毕，准确率: 69.14 %

===== 阶段 3: 提取 C 语言测试向量 (单张图片) =====
✅ 完成！test_vector.h 已生成，Multiplier=151
